## LangChain

Definition: LangChain is a framework that helps developers build AI applications by connecting LLMs with data, tools, and APIs.

example: Suppose you want to build an AI chatbot that reads your PDF and answers questions. LangChain helps you connect the AI model with your PDF.

## why we used?

Definition: We use LangChain to make it easier to build AI applications without writing everything from scratch.

Example:
- Connect AI models like OpenAI and Groq.
- Give instructions to AI using prompts.
- Connect AI with PDF files and databases.
- Allow AI to use external tools and APIs.

## Coding

In [10]:
pip install langchain

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.5/711.5 kB 14.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 15.0 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 14.5 MB/s  0:00:00
Using cached httpx-0.28.1-py3-no

In [13]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [20]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [21]:
from groq import Groq
import langchain
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
secret_groq_key = os.getenv("GROQ_API_KEY")

In [33]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [34]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="openai/gpt-oss-120b",
)

print(chat_completion.choices[0].message.content)

## Why Speed Matters for Modern Language Models  

| **Domain** | **What “fast” means** | **Why it matters** |
|------------|----------------------|--------------------|
| **Interactive AI (chatbots, voice assistants, search)** | Sub‑second latency per request (often < 100 ms) | • Users abandon a conversation if they have to wait. <br>• Real‑time feedback is essential for voice‑driven or AR/VR experiences. |
| **Enterprise‑scale services** | Thousands‑to‑millions of queries per second, cost‑per‑token low | • High throughput reduces cloud‑bill spikes. <br>• Predictable latency enables Service‑Level Agreements (SLAs). |
| **Edge / Mobile deployment** | Low‑power inference on CPUs, NPUs, or micro‑controllers | • Battery life, heat dissipation, and network bandwidth are limited. <br>• Offline capability (e.g., on‑device translation) improves privacy. |
| **Research & Development** | Fast training loops, rapid hyper‑parameter sweeps | • Shorter experiment cycles accelerate scientific progre

# Try documents code and implements with help of groq

In [39]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json
import os
from dotenv import load_dotenv

In [40]:
# Load Groq API key
load_dotenv()
secret_groq_key = os.getenv("GROQ_API_KEY")

In [41]:
# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    groq_api_key=secret_groq_key
)

In [42]:
# Define the expected JSON structure
parser = JsonOutputParser(pydantic_object={
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "price": {"type": "number"},
        "features": {
            "type": "array",
            "items": {"type": "string"}
        }
    }
})

In [46]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="Product name")
    price: float = Field(description="Product price without currency")
    features: list[str] = Field(description="Product features")
    

parser = JsonOutputParser(pydantic_object=Product)    

In [47]:
# Create a prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract product details into JSON with this structure:
    {{
        "name": "product name here",
        "price": number_here_without_currency_symbol,
        "features": ["feature1", "feature2", "feature3"]
    }}
    
    {format_instructions}"""),
    ("user", "{input}")
]).partial(format_instructions=parser.get_format_instructions())


In [48]:
# Create the chain
chain = prompt | llm | parser

def parse_product(description: str) -> dict:
    result = chain.invoke({"input": description})
    print(json.dumps(result, indent=2))
    return result


In [49]:
# Example usage
description = """The Kees Van Der Westen Speedster is a high-end,
single-group espresso machine known for its precision and industrial design.
It features dual boilers, PID temperature control, and a pre-infusion system.
Its price is approximately $14,499."""

parse_product(description)

{
  "name": "Kees Van Der Westen Speedster",
  "price": 14499,
  "features": [
    "high-end",
    "single-group espresso machine",
    "precision",
    "industrial design",
    "dual boilers",
    "PID temperature control",
    "pre-infusion system"
  ]
}


{'name': 'Kees Van Der Westen Speedster',
 'price': 14499,
 'features': ['high-end',
  'single-group espresso machine',
  'precision',
  'industrial design',
  'dual boilers',
  'PID temperature control',
  'pre-infusion system']}

# Chaining system

In [50]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [51]:
# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

In [52]:
# Create prompt
prompt = ChatPromptTemplate.from_template("""
Previous Answer:
{previous_answer}

Current Question:
{question}

Answer the current question using the previous answer as context.
If needed, add new information to explain the topic clearly.
""")

In [54]:
# Create chain
chain = prompt | llm | StrOutputParser()

# Initial question
question = "What is Python?"
previous_answer = "No previous context."

# Sequential execution
questions = [
    question,
    "Why do we use Python?",
    "What are the main features of Python?",
    "Where is Python used?",
    "What are the limitations of Python?"
]


In [55]:
for i, q in enumerate(questions, 1):

    response = chain.invoke({
        "previous_answer": previous_answer,
        "question": q
    })

    print(f"\nQuestion {i}: {q}")
    print(f"Answer {i}: {response}")

    # Pass current answer to the next step
    previous_answer = response



Question 1: What is Python?
Answer 1: Since there’s no prior answer to build on, here’s a fresh, concise overview of Python:

**Python** is a high‑level, interpreted programming language known for its readability, simplicity, and versatility. Created by Guido van Rossum and first released in 1991, Python’s design philosophy emphasizes clear, human‑friendly code (e.g., “readability counts” is one of the guiding principles in the *Zen of Python*).

### Core Characteristics
| Feature | Why It Matters |
|---------|----------------|
| **Readable Syntax** | Uses indentation to define code blocks, reducing the need for braces or keywords and making code easier to understand. |
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter, allowing rapid development and immediate feedback. |
| **Dynamic Typing** | Variables don’t need explicit type declarations; types are inferred at runtime. |
| **Rich Standard Library** | Includes modules for file I/O, networking, web services









---






## Small System 

So based on that prompt chaining system. we build small system those can 
1) take a suggested name for e-commerse site
2) based on name they suggest what we can sell 

In [57]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

In [58]:
# Load API key
load_dotenv()

# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [60]:
# Create prompt
prompt = ChatPromptTemplate.from_template("""
Previous Result:
{previous_result}

Task:
{question}

Generate a useful and creative result.
""")

# Create chain
chain = prompt | llm | StrOutputParser()

# Define questions
questions = [
    "Suggest one creative and unique e-commerce store name.",
    "Based on this store name, suggest what types of products this e-commerce store should sell. Give 5 product categories."
]

# Store previous result
previous_result = "No previous result."

In [61]:
# Execute sequential chain
for i, question in enumerate(questions, 1):

    result = chain.invoke({
        "previous_result": previous_result,
        "question": question
    })

    # Uncomment below to see every step
    # print(f"\nStep {i}:")
    # print(result)

    previous_result = result

# Display only final result
print("\nFinal E-commerce Store Recommendation:")
print(previous_result)


Final E-commerce Store Recommendation:
**LuminaraMarket – Suggested Product Categories**

| # | Category | Why it fits the “Luminara” vibe |
|---|----------|---------------------------------|
| 1 | **Home Lighting & Fixtures** | Stylish LED lamps, smart bulbs, chandeliers, floor‑lights, and decorative lanterns that turn any space into a glowing haven. |
| 2 | **Wellness & Mood‑Enhancing Gadgets** | Aromatherapy diffusers with ambient light, sunrise‑simulation alarm clocks, color‑changing sound‑bars, and sleep‑aid devices that blend light and sound for relaxation. |
| 3 | **Outdoor & Garden Illumination** | Solar string lights, waterproof spotlights, fire‑fly lanterns, illuminated planters, and patio glow‑up kits for creating magical evenings outdoors. |
| 4 | **Tech‑Enhanced Wearables** | Light‑up smart jewelry, LED‑infused active‑wear, reflective safety gear, and programmable accessories that let customers “wear the light.” |
| 5 | **Creative DIY Light Kits** | Build‑your‑own neon si

---

## Sequential Chain?

Definition: Sequential Chain is a technique in LangChain where multiple tasks are executed one after another, and the output of one task is passed as input to the next task.

Simple example:

Suppose you want to build an e-commerce recommendation system.
1. Task 1: Generate an e-commerce store name.
2. Task 2: Suggest product categories based on the store name.
3. Task 3: Suggest products based on those categories.

Why do we use Sequential Chain?
- To execute multiple tasks in a specific order.
- To pass output from one task to another.
- To build multi-step AI workflows.
- To break complex tasks into smaller tasks.

#### step1: Import libraries and initialize LLM

In [62]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

#### step2: Create three prompts

In [63]:
# Task 1: Generate store name
prompt1 = ChatPromptTemplate.from_template(
    "Suggest one creative e-commerce store name for {business}."
)

# Task 2: Generate product categories
prompt2 = ChatPromptTemplate.from_template(
    "For the e-commerce store {store_name}, suggest 5 product categories."
)

# Task 3: Suggest products
prompt3 = ChatPromptTemplate.from_template(
    "For these product categories: {categories}, suggest 3 products for each category."
)

#### step3: Create individual chains

In [64]:
chain1 = prompt1 | llm | StrOutputParser()
chain2 = prompt2 | llm | StrOutputParser()
chain3 = prompt3 | llm | StrOutputParser()

#### Step 4: Execute sequentially

In [65]:
# First chain
store_name = chain1.invoke({
    "business": "Online fashion store"
})
# Second chain uses first chain's output
categories = chain2.invoke({
    "store_name": store_name
})
# Third chain uses second chain's output
products = chain3.invoke({
    "categories": categories
})

# Final output
print(products)

Below is a ready‑to‑use product‑list you can copy straight into your store‑management system (CSV, spreadsheet, Shopify bulk‑import, etc.).  
Each row contains a **product name**, a short **description**, key **specs**, an **estimated price range**, and a **SKU** that follows a simple naming convention (CATEGORY‑##‑VARIANT).

| # | Category | Product | Description (key selling points) | Specs / Options | Price (USD) | SKU |
|---|----------|---------|----------------------------------|----------------|-------------|-----|
| 1 | **Signature Canvas Prints** | **Midnight Skyline – 24×36”** | Moody city‑scape in deep blues & golds; ready‑to‑hang with hidden‑mount hardware. | Framed (black wood) / Unframed; matte finish | $79 – $119 | SIG‑01‑F |
| 1 | **Signature Canvas Prints** | **Botanical Burst – 30×30”** | Vibrant, oversize tropical leaves printed on premium 13 mil cotton canvas; UV‑protected inks. | Unframed only; gallery‑wrap | $69 – $99 | SIG‑02‑U |
| 1 | **Signature Canvas Prints** 